In [1]:
!pip -q install datasets transformers accelerate scikit-learn

In [2]:
import pandas as pd
from datasets import load_dataset, Dataset
from sklearn.metrics import accuracy_score, classification_report
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

In [3]:
dataset = load_dataset("SetFit/amazon_reviews_multi_en")
train_data = dataset["train"].to_pandas()
test_data = dataset["test"].to_pandas()

def get_sentiment(rating):
    if rating <= 1:
        return 0
    elif rating == 2:
        return 1
    return 2

train_data["sentiment"] = train_data["label"].apply(get_sentiment)
test_data["sentiment"] = test_data["label"].apply(get_sentiment)

README.md:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

train.jsonl: reconstructing file:   0%|          |  0.00B / 47.4MB            

train.jsonl: downloading bytes:           |  0.00B            

validation.jsonl:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/1.18M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/200000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [4]:
train_data = pd.concat([
    train_data[train_data["sentiment"] == i].sample(100, random_state=42)
    for i in range(3)
]).reset_index(drop=True)

test_data = pd.concat([
    test_data[test_data["sentiment"] == i].sample(50, random_state=42)
    for i in range(3)
]).reset_index(drop=True)

train_text = train_data["text"].astype(str)
train_labels = train_data["sentiment"].astype(int)
test_text = test_data["text"].astype(str)
test_labels = test_data["sentiment"].astype(int)

sentiment_names = ["Negative", "Neutral", "Positive"]

print("Training reviews:", len(train_data))
print("Testing reviews:", len(test_data))

Training reviews: 300
Testing reviews: 150


In [5]:
text_tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
text_tokenizer.fit_on_texts(train_text)

train_sequences = pad_sequences(
    text_tokenizer.texts_to_sequences(train_text), maxlen=100
)
test_sequences = pad_sequences(
    text_tokenizer.texts_to_sequences(test_text), maxlen=100
)

lstm_model = Sequential([
    Embedding(10000, 64),
    LSTM(32),
    Dense(3, activation="softmax")
])

lstm_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

lstm_model.fit(
    train_sequences, train_labels,
    epochs=3, batch_size=32, verbose=1
)

lstm_predictions = lstm_model.predict(test_sequences, verbose=0).argmax(axis=1)
lstm_accuracy = accuracy_score(test_labels, lstm_predictions)

print("\nLSTM Accuracy:", round(lstm_accuracy, 4))
print(classification_report(
    test_labels, lstm_predictions,
    labels=[0, 1, 2], target_names=sentiment_names, zero_division=0
))

Epoch 1/3
10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 119ms/step - accuracy: 0.3267 - loss: 1.0992
Epoch 2/3
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 94ms/step - accuracy: 0.6467 - loss: 1.0816
Epoch 3/3
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 100ms/step - accuracy: 0.8433 - loss: 1.0556

LSTM Accuracy: 0.4133
              precision    recall  f1-score   support

    Negative       0.48      0.26      0.34        50
     Neutral       0.35      0.36      0.36        50
    Positive       0.43      0.62      0.51        50

    accuracy                           0.41       150
   macro avg       0.42      0.41      0.40       150
weighted avg       0.42      0.41      0.40       150



In [6]:
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

bert_train_data = Dataset.from_pandas(pd.DataFrame({
    "text": train_text,
    "labels": train_labels
}))
bert_test_data = Dataset.from_pandas(pd.DataFrame({
    "text": test_text,
    "labels": test_labels
}))

def tokenize_reviews(reviews):
    return bert_tokenizer(
        reviews["text"],
        truncation=True,
        padding="max_length",
        max_length=96
    )

bert_train_data = bert_train_data.map(tokenize_reviews, batched=True)
bert_test_data = bert_test_data.map(tokenize_reviews, batched=True)

bert_train_data = bert_train_data.remove_columns(["text"])
bert_test_data = bert_test_data.remove_columns(["text"])
bert_train_data.set_format("torch")
bert_test_data.set_format("torch")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

In [7]:
bert_model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=3
)

training_args = TrainingArguments(
    output_dir="./bert_results",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    report_to="none",
    save_strategy="no",
    dataloader_num_workers=0
)

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=bert_train_data
)

trainer.train()

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3

Step,Training Loss


TrainOutput(global_step=150, training_loss=1.0994110107421875, metrics={'train_runtime': 479.7537, 'train_samples_per_second': 0.625, 'train_steps_per_second': 0.313, 'total_flos': 14800129747200.0, 'train_loss': 1.0994110107421875, 'epoch': 1.0})

In [8]:
bert_predictions = trainer.predict(bert_test_data).predictions.argmax(axis=1)
bert_accuracy = accuracy_score(test_labels, bert_predictions)

print("\nBERT Accuracy:", round(bert_accuracy, 4))
print(classification_report(
    test_labels, bert_predictions,
    labels=[0, 1, 2], target_names=sentiment_names, zero_division=0
))


BERT Accuracy: 0.5067
              precision    recall  f1-score   support

    Negative       0.61      0.28      0.38        50
     Neutral       0.39      0.74      0.51        50
    Positive       0.78      0.50      0.61        50

    accuracy                           0.51       150
   macro avg       0.59      0.51      0.50       150
weighted avg       0.59      0.51      0.50       150



In [9]:
print("\nMODEL COMPARISON")
print("LSTM Accuracy:", round(lstm_accuracy, 4))
print("BERT Accuracy:", round(bert_accuracy, 4))

if bert_accuracy > lstm_accuracy:
    print("BERT achieved higher accuracy on this test sample.")
elif lstm_accuracy > bert_accuracy:
    print("LSTM achieved higher accuracy on this test sample.")
else:
    print("Both models achieved the same accuracy on this test sample.")


MODEL COMPARISON
LSTM Accuracy: 0.4133
BERT Accuracy: 0.5067
BERT achieved higher accuracy on this test sample.


## 10. Discussion

**Transfer learning:** BERT begins with language knowledge learned during pre-training and fine-tunes its model for sentiment classification.

**Model comparison:** Compare the accuracy and class-wise precision, recall, and F1-scores printed above. Which model performs better depends on the actual results.

**Deployment constraints:** BERT generally uses more memory and computing power than a small LSTM. Deployment decisions should consider inference speed, available hardware, memory, and cost.

**Note:** The smaller sample is for a manageable demonstration. Results may change with larger training data or different random seeds.